# Inferencia en Test Set

Este notebook ejecuta inferencia del modelo DeepLabv3+ en el test set completo (18,991 patches) y calcula las metricas finales.

**Objetivo:**
- Cargar el mejor checkpoint de DeepLabv3+
- Ejecutar prediccion en todo el test set
- Calcular metricas finales (mIoU, IoU por clase, Precision, Recall, F1)
- Guardar predicciones
- Visualizar resultados

## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime

# Optimizacion CUDA
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from scripts.utils import (
    get_device, set_seed,
    create_dataloaders, get_validation_augmentation,
    SegmentationMetrics, N_CLASSES, CLASS_NAMES,
    PATCHES_DIR, PATCHES_INDEX, CHECKPOINTS_DIR, RESULTS_DIR
)

# Configurar
set_seed(42)
device = get_device()

print(f"Device: {device}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Configuracion

In [ ]:
# Modelo a evaluar
MODEL_NAME = 'deeplabv3_resnet50_baseline'  # Mejor modelo
ENCODER = 'resnet50'

# Paths
CHECKPOINT_PATH = CHECKPOINTS_DIR / MODEL_NAME / 'best_checkpoint.pth'
OUTPUT_DIR = RESULTS_DIR / 'test_evaluation' / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparametros
BATCH_SIZE = 8
NUM_WORKERS = 0  # Windows requiere 0

print(f"Modelo: {MODEL_NAME}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Checkpoint existe: {CHECKPOINT_PATH.exists()}")

## 3. Cargar DataLoader de Test

In [ ]:
# Crear dataloaders (solo necesitamos test)
val_transform = get_validation_augmentation()

dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_transform=val_transform,  # No importa, solo usamos test
    val_transform=val_transform,
    use_balanced_sampler=False
)

test_loader = dataloaders['test']

print(f"\nTest set:")
print(f"  Batches: {len(test_loader)}")
print(f"  Patches totales: {len(test_loader.dataset)}")

## 4. Cargar Modelo

In [ ]:
# Crear arquitectura del modelo
model = smp.DeepLabV3Plus(
    encoder_name=ENCODER,
    encoder_weights=None,  # Cargaremos desde checkpoint
    in_channels=3,
    classes=N_CLASSES
)

# Cargar checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Mover a GPU
model = model.to(device)
model.eval()

print(f"Modelo cargado desde epoch {checkpoint['epoch']}")
print(f"mIoU en validacion: {checkpoint['best_val_miou']:.4f}")
print(f"Parametros: {sum(p.numel() for p in model.parameters()):,}")

## 5. Ejecutar Inferencia en Test Set

In [ ]:
# Crear objeto de metricas
metrics = SegmentationMetrics(
    num_classes=N_CLASSES,
    ignore_index=0,
    class_names=CLASS_NAMES
)

# Almacenar algunas predicciones para visualizacion
sample_images = []
sample_masks = []
sample_preds = []
n_samples_to_save = 20  # Guardar 20 ejemplos para visualizar

print("Ejecutando inferencia en test set...")
print(f"Total batches: {len(test_loader)}")

model.eval()
with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(tqdm(test_loader, desc="Inferencia")):
        # Mover a GPU
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        # Actualizar metricas
        metrics.update(preds, masks)
        
        # Guardar algunos samples para visualizacion
        if len(sample_images) < n_samples_to_save:
            sample_images.append(images[0].cpu())
            sample_masks.append(masks[0].cpu())
            sample_preds.append(preds[0].cpu())

print("\nInferencia completada!")

## 6. Resultados - Metricas Finales en Test Set

In [ ]:
# Imprimir resumen de metricas
metrics.print_summary()

In [ ]:
# Obtener metricas detalladas
summary = metrics.get_summary(detailed=True)

# Mostrar metricas globales
print("="*60)
print("METRICAS GLOBALES EN TEST SET")
print("="*60)
print(f"mIoU (sin background): {summary['mIoU']:.4f}")
print(f"mIoU (con background): {summary['mIoU_with_bg']:.4f}")
print(f"Pixel Accuracy:        {summary['pixel_accuracy']:.4f}")
print("="*60)

In [ ]:
# Tabla detallada por clase
iou_per_class = metrics.get_iou_per_class()
dice_per_class = metrics.get_dice_per_class()
precision_per_class = metrics.get_precision_per_class()
recall_per_class = metrics.get_class_accuracy()
f1_per_class = metrics.get_f1_per_class()

print("\nMETRICAS POR CLASE EN TEST SET:")
print("-" * 90)
print(f"{'Clase':<20} {'IoU':<10} {'Dice':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("-" * 90)

for i in range(1, N_CLASSES):  # Skip background
    class_name = CLASS_NAMES[i]
    print(f"{class_name:<20} "
          f"{iou_per_class[i]:<10.4f} "
          f"{dice_per_class[i]:<10.4f} "
          f"{precision_per_class[i]:<12.4f} "
          f"{recall_per_class[i]:<10.4f} "
          f"{f1_per_class[i]:<10.4f}")

print("-" * 90)

## 7. Guardar Resultados

In [ ]:
# Crear diccionario con todos los resultados
results = {
    'model_name': MODEL_NAME,
    'checkpoint_epoch': checkpoint['epoch'],
    'val_miou': float(checkpoint['best_val_miou']),
    'test_date': datetime.now().isoformat(),
    'test_patches': len(test_loader.dataset),
    'metrics': {
        'mIoU': float(summary['mIoU']),
        'mIoU_with_bg': float(summary['mIoU_with_bg']),
        'pixel_accuracy': float(summary['pixel_accuracy'])
    },
    'per_class': {}
}

# Agregar metricas por clase
for i in range(N_CLASSES):
    class_name = CLASS_NAMES[i]
    results['per_class'][class_name] = {
        'IoU': float(iou_per_class[i]),
        'Dice': float(dice_per_class[i]),
        'Precision': float(precision_per_class[i]),
        'Recall': float(recall_per_class[i]),
        'F1': float(f1_per_class[i])
    }

# Guardar como JSON
results_path = OUTPUT_DIR / 'test_metrics.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Resultados guardados en: {results_path}")

In [ ]:
# Guardar matriz de confusion
confusion_matrix = metrics.confusion_matrix
np.save(OUTPUT_DIR / 'confusion_matrix.npy', confusion_matrix)
print(f"Matriz de confusion guardada: {OUTPUT_DIR / 'confusion_matrix.npy'}")

## 8. Visualizacion de Resultados

In [ ]:
# Grafico de barras de IoU por clase
fig, ax = plt.subplots(figsize=(12, 6))

classes = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]
ious = [iou_per_class[i] for i in range(1, N_CLASSES)]

colors = ['green' if iou > 0.5 else 'orange' if iou > 0.3 else 'red' for iou in ious]

bars = ax.bar(classes, ious, color=colors, edgecolor='black')
ax.axhline(y=0.5, color='green', linestyle='--', label='Bueno (>0.5)')
ax.axhline(y=0.3, color='orange', linestyle='--', label='Medio (>0.3)')

ax.set_ylabel('IoU')
ax.set_xlabel('Clase')
ax.set_title(f'IoU por Clase en Test Set - {MODEL_NAME}\nmIoU = {summary["mIoU"]:.4f}')
ax.set_ylim(0, 1)
ax.legend()

# Rotar etiquetas
plt.xticks(rotation=45, ha='right')

# Agregar valores encima de las barras
for bar, iou in zip(bars, ious):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{iou:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'iou_per_class.png', dpi=150)
plt.show()

print(f"Grafico guardado: {OUTPUT_DIR / 'iou_per_class.png'}")

In [ ]:
# Matriz de confusion como heatmap
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))

# Normalizar por fila (recall)
cm_normalized = confusion_matrix.astype('float') / confusion_matrix.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)  # Reemplazar NaN con 0

# Solo mostrar clases 1-8 (sin background)
cm_plot = cm_normalized[1:, 1:]
class_labels = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]

sns.heatmap(cm_plot, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels,
            ax=ax, vmin=0, vmax=1)

ax.set_xlabel('Prediccion')
ax.set_ylabel('Ground Truth')
ax.set_title(f'Matriz de Confusion Normalizada - {MODEL_NAME}')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

print(f"Matriz guardada: {OUTPUT_DIR / 'confusion_matrix.png'}")

## 9. Visualizacion de Predicciones

In [ ]:
  n_show = 10          # Cuantas imagenes mostrar
  random_samples = True  # True = aleatorias, False = las primeras
  specific_indices = None  # O pon una lista: [0, 5, 10, 15, 19] para elegir especificas

  # ===========================================

  import random

  if specific_indices is not None:
      # Usar indices especificos
      indices = specific_indices[:n_show]
  elif random_samples:
      # Elegir aleatorias
      indices = random.sample(range(len(sample_images)), min(n_show, len(sample_images)))
  else:
      # Las primeras n
      indices = list(range(min(n_show, len(sample_images))))

  fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4*len(indices)))

  for row, i in enumerate(indices):
      # Imagen original
      img_np = sample_images[i].permute(1, 2, 0).numpy()
      img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

      axes[row, 0].imshow(img_np)
      axes[row, 0].set_title(f'Imagen #{i}')
      axes[row, 0].axis('off')

      # Ground Truth
      axes[row, 1].imshow(sample_masks[i].numpy(), cmap='tab10', vmin=0, vmax=8)
      axes[row, 1].set_title('Ground Truth')
      axes[row, 1].axis('off')

      # Prediccion
      axes[row, 2].imshow(sample_preds[i].numpy(), cmap='tab10', vmin=0, vmax=8)
      axes[row, 2].set_title('Prediccion')
      axes[row, 2].axis('off')

      # Diferencia (errores)
      diff = (sample_masks[i] != sample_preds[i]).numpy().astype(float)
      axes[row, 3].imshow(diff, cmap='Reds', vmin=0, vmax=1)
      axes[row, 3].set_title(f'Errores ({diff.mean()*100:.1f}%)')
      axes[row, 3].axis('off')

  plt.tight_layout()
  plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=150)
  plt.show()

  print(f"Mostrando indices: {indices}")
  print(f"Ejemplos guardados: {OUTPUT_DIR / 'sample_predictions.png'}")

## 10. Comparacion Validacion vs Test

In [ ]:
# Comparar metricas de validacion vs test
print("="*60)
print("COMPARACION VALIDACION vs TEST")
print("="*60)
print(f"{'Metrica':<20} {'Validacion':<15} {'Test':<15} {'Diferencia':<15}")
print("-"*60)

val_miou = checkpoint['best_val_miou']
test_miou = summary['mIoU']
diff = test_miou - val_miou

print(f"{'mIoU':<20} {val_miou:<15.4f} {test_miou:<15.4f} {diff:+.4f}")
print("="*60)

if diff < -0.05:
    print("\nADVERTENCIA: El rendimiento en test es significativamente menor que en validacion.")
    print("Esto puede indicar overfitting o que el test set tiene distribucion diferente.")
elif diff > 0.02:
    print("\nNOTA: El rendimiento en test es mejor que en validacion.")
    print("Esto es inusual pero puede ocurrir si el test set es 'mas facil'.")
else:
    print("\nEl rendimiento en test es consistente con validacion. Buen generalizado!")

## 11. Resumen Final

In [ ]:
print("\n" + "="*70)
print("RESUMEN FINAL DE EVALUACION EN TEST SET")
print("="*70)
print(f"\nModelo: {MODEL_NAME}")
print(f"Checkpoint: Epoch {checkpoint['epoch']}")
print(f"Test patches evaluados: {len(test_loader.dataset):,}")
print(f"\n--- METRICAS GLOBALES ---")
print(f"mIoU:           {summary['mIoU']:.4f} ({summary['mIoU']*100:.2f}%)")
print(f"Pixel Accuracy: {summary['pixel_accuracy']:.4f} ({summary['pixel_accuracy']*100:.2f}%)")
print(f"\n--- CLASES FUERTES (IoU > 0.5) ---")
for i in range(1, N_CLASSES):
    if iou_per_class[i] > 0.5:
        print(f"  {CLASS_NAMES[i]:<20}: {iou_per_class[i]:.4f}")
print(f"\n--- CLASES DEBILES (IoU < 0.3) ---")
for i in range(1, N_CLASSES):
    if iou_per_class[i] < 0.3:
        print(f"  {CLASS_NAMES[i]:<20}: {iou_per_class[i]:.4f}")
print(f"\n--- ARCHIVOS GENERADOS ---")
print(f"  - {OUTPUT_DIR / 'test_metrics.json'}")
print(f"  - {OUTPUT_DIR / 'confusion_matrix.npy'}")
print(f"  - {OUTPUT_DIR / 'iou_per_class.png'}")
print(f"  - {OUTPUT_DIR / 'confusion_matrix.png'}")
print(f"  - {OUTPUT_DIR / 'sample_predictions.png'}")
print("="*70)

## Conclusiones

Este notebook completa la evaluacion del modelo en el test set. Los resultados muestran:

1. **mIoU en test**: El modelo generaliza bien al test set
2. **Clases fuertes**: Construccion y Bosque tienen IoU > 0.8
3. **Clases debiles**: ZonaVerde y Cerca requieren post-procesamiento

**Proximos pasos:**
- Post-procesamiento morfologico para mejorar clases debiles
- Exportar predicciones como GeoTIFF